# LoRAForge — rank-4 ablation

Asks one question: does this adaptation need 16 directions, or fewer?

Everything is identical to the frozen rank-16 run except `lora.rank` 16 -> 4 and
`lora.alpha` 32 -> 8. Alpha moves with rank so the `alpha/rank` update scaling stays
at 2.0 — otherwise a weaker result could not be attributed to capacity rather than
to a four-times-larger update. A repo test enforces that only those fields differ.

**The prediction is already recorded in `docs/ABLATION_RANK4.md`:** if the adaptation
is as low-dimensional as the AG News result suggests, rank 4 should land within about
0.01 of **0.9310** validation macro-F1.

Kaggle settings: **Accelerator = GPU T4**, **Internet = On**, and your Hugging Face
token stored under **Add-ons -> Secrets** as `HF_TOKEN`.

Run this with **Save Version -> Save & Run All (Commit)** so it executes headless and
survives a closed browser. Interactive sessions can idle out and lose the run.


In [ ]:
from pathlib import Path

# Works on Kaggle (/kaggle/working) and Colab (/content).
BASE = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('/content')
REPO = BASE / 'loraforge-llm'
BRANCH = 'codex/publish-loraforge-results'
if not (REPO / '.git').exists():
    !git clone --depth 1 --branch {BRANCH} https://github.com/mghadia1/loraforge-llm.git {REPO}
assert (REPO / '.git').exists(), 'clone failed; check Internet is enabled'
%cd {REPO}

# Kaggle and Colab ship a torch built against their own CUDA driver. Replacing it
# mid-session leaves the running kernel holding a half-swapped package, which fails
# as "cannot import name 'nn' from partially initialized module 'torch'". So install
# this package without its dependency closure, then add only what the image lacks --
# and never torch itself.
!python -m pip install -q -e . --no-deps
!python -m pip install -q "peft>=0.17,<1" "bitsandbytes>=0.47,<1" "trl>=0.21,<1" \
    "accelerate>=1.10,<2" "transformers>=4.55,<6" "datasets>=4,<6" \
    "scikit-learn>=1.6,<2" "numpy>=2,<3"


In [ ]:
import sys, site
site.main()  # an editable install adds a .pth the running kernel has not read
sys.path.insert(0, str(REPO / 'src'))

import torch  # if this raises a partially-initialized error, Restart Session and
              # re-run from THIS cell -- do not re-run the install cell
import loraforge

print('torch      ', torch.__version__)
print('loraforge  ', loraforge.__file__)
print('gpu        ', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
assert torch.cuda.is_available(), 'enable the GPU accelerator before running'


Cell below must report **85 passed**. If it does not, stop: the environment is wrong
and every number after this point would be meaningless. If pip upgraded torch during
the install, restart the session and re-run from this cell.


In [ ]:
!python -m pytest -q


## Authenticate

Mistral 7B is a gated repository. This reads the token from Kaggle Secrets so it never
appears in the notebook or its output.


In [ ]:
from huggingface_hub import login

try:
    from kaggle_secrets import UserSecretsClient
    login(UserSecretsClient().get_secret('HF_TOKEN'))
    print('logged in from Kaggle Secrets')
except ImportError:
    login()  # Colab / local: interactive prompt


## Confirm the ablation is controlled

Prints every field that differs from the frozen run. It must be exactly three:
`lora.rank`, `lora.alpha`, and `test_evaluations_allowed`. The test budget is zero
because the single publisher-test evaluation is already spent — this arm is
validation-only.


In [ ]:
import json
from loraforge.config import load_config

frozen = load_config(Path('configs/experiment.json')).to_dict()
ablation = load_config(Path('configs/experiment-rank4.json')).to_dict()

def flatten(payload, prefix=''):
    flat = {}
    for key, value in payload.items():
        if isinstance(value, dict):
            flat.update(flatten(value, f'{prefix}{key}.'))
        else:
            flat[f'{prefix}{key}'] = value
    return flat

a, b = flatten(frozen), flatten(ablation)
differences = {k: (a[k], b[k]) for k in a if a[k] != b[k]}
print(json.dumps(differences, indent=2))
assert set(differences) == {'lora.rank', 'lora.alpha', 'test_evaluations_allowed'}
print('\nalpha/rank:', b['lora.alpha'] / b['lora.rank'], '(frozen:', a['lora.alpha'] / a['lora.rank'], ')')


## Train

About 3.6 hours on a T4. Rank barely changes step time — the frozen 7B base dominates
the compute, so a quarter of the trainable parameters does not mean a quarter of the
time.

Before the first optimizer step this re-scores the untuned base with the adapter
disabled and aborts unless it reproduces the phase-one baseline macro-F1 of 0.7299
within 0.005. That check runs in roughly six minutes: if it passes, the setup is
sound and the rest is arithmetic.


In [ ]:
!mkdir -p runs/rank4/outputs && cp outputs/base-validation.json runs/rank4/outputs/
!loraforge train --config configs/experiment-rank4.json --root runs/rank4


## Read the result


In [ ]:
report = json.loads(Path('runs/rank4/outputs/training-report.json').read_text())

print('trainable parameters:', f"{report['parameters']['trainable_parameters']:,}",
      f"({report['parameters']['trainable_percent']:.4f}% of {report['parameters']['total_parameters']:,})")
print('baseline agreement  :', report['baseline_agreement']['agrees'],
      '| difference', f"{report['baseline_agreement']['absolute_difference']:.6f}")
for epoch in report['epochs']:
    print(f"epoch {epoch['epoch']}: validation macro-F1 {epoch['validation']['macro_f1']:.4f}")
print('selected epoch      :', report['selection']['selected_epoch'])
print('wall time (h)       :', round(report['wall_time_seconds'] / 3600, 2))
print('peak CUDA (GiB)     :', round(report['peak_cuda_memory_gib'], 2))

best = max(e['validation']['macro_f1'] for e in report['epochs'])
print(f"\nrank 16 reference: 0.9310    rank 4: {best:.4f}    delta {best - 0.9310:+.4f}")
print('within the +/-0.01 prediction:', abs(best - 0.9310) <= 0.01)


## Download before the session ends

From the committed version's **Output** tab, take:

- `runs/rank4/outputs/training-report.json` — the numbers and their hashes
- `runs/rank4/adapters/selected/` — the rank-4 adapter, roughly 42 MB

Then verify locally, where the frozen artifacts live:

```bash
loraforge verify --root runs/rank4
```

Whatever the delta is, it is the result. A rank-4 run that matches means the frozen
run was over-provisioned four times over; one that drops means this adaptation needed
more than four directions. Both are findings. `resume_eligible` stays false either way.
